# Maya 2 Native — Indic TTS (Colab)

Runs [`maya-research/maya2-native`](https://huggingface.co/maya-research/maya2-native): streaming TTS for
11 Indic languages + English, 20 built-in voices, zero-shot cross-lingual voice cloning, Apache-2.0.

**This needs a GPU runtime.** The inference engine is hard CUDA — CUDA graphs, Triton attention kernels,
pinned-memory transfers. There is no CPU path, so a CPU runtime cannot run it at all.

Set **Runtime → Change runtime type → L4 GPU** (or A100). Notes on the free **T4**: it is Turing (sm_75) and
does not support bf16 natively, while this model ships a bf16 backbone — cell 1 checks this and tells you
where you stand. VRAM needed is about **9.2 GB peak while loading, 7.7 GB steady** at `max_seats=1`.

Run the cells top to bottom. Cell 2 reinstalls torch and **requires a runtime restart** before cell 3.

In [ ]:
#@title 1 · Check the runtime can actually run this
import subprocess, sys

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or
      'no NVIDIA GPU visible — switch to a GPU runtime, this model cannot run on CPU')
try:
    import torch
    if torch.cuda.is_available():
        cc = torch.cuda.get_device_capability()
        free, total = (v / 1e9 for v in torch.cuda.mem_get_info())
        print(f'compute capability {cc[0]}.{cc[1]}  |  bf16 native: {torch.cuda.is_bf16_supported()}')
        print(f'VRAM {free:.1f} GB free of {total:.1f} GB  (needs ~9.2 GB peak at max_seats=1)')
        if cc[0] < 8:
            print('WARNING: pre-Ampere GPU. The bf16 backbone and the Triton kernels are built for sm_80+;\n'
                  '         expect slow emulation or hard kernel failures. Prefer an L4 or A100 runtime.')
        if free < 9.5:
            print('WARNING: tight on VRAM. Keep max_seats=1 and restart the runtime if loading OOMs.')
    else:
        print('torch sees no CUDA device.')
except ImportError:
    print('torch not installed yet — that is fine, cell 2 installs it.')

In [ ]:
#@title 2 · Install  (reinstalls torch — RESTART THE RUNTIME after this cell)
# maya3 pins torch==2.8.0 + torchaudio==2.8.0 (cu128) and transformers==5.6.2, so this replaces
# Colab's preinstalled torch. ~3 GB of wheels; takes a few minutes.
!pip install -q "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" \
    --extra-index-url https://download.pytorch.org/whl/cu128
print('\ninstalled — now Runtime -> Restart session, then continue from cell 3')

In [ ]:
#@title 3 · Load the model
MAX_SEATS = 1  #@param {type:"integer"}

import soundfile as sf, numpy as np, IPython.display as ipd
from maya3 import Maya3TTS

# Weights (~5 GB), the Mimi codec and the ECAPA speaker encoder download on first use.
tts = Maya3TTS.from_pretrained('maya-research/maya2-native', max_seats=MAX_SEATS)
print('loaded')

## Generate

`lang` is one of `as bn en gu hi kn ml mr or pa ta te`. `normalize=True` is **required on the Python API**
(the HTTP/WebSocket servers default it on) — it expands numbers, currency, dates, times, URLs and email
addresses into spoken form. Without it those are read as raw characters.

Voices with at least an hour of **Bengali** training audio: `Ahana Vaidehi Malini Charvi Jahnavi Anirban
Nakul Vasant Siddhant Satvik`. The other ten (`Abhay Nivedita Ishwari Ritvik Vrinda Sreekar Kayal Shaunak
Akshat Tanushree`) are en/hi/ta/te-trained — they will still speak Bengali, but cross-lingually, carrying
their training accent.

In [ ]:
#@title 4 · Speak
TEXT  = "\u0986\u0997\u09be\u09ae\u09c0\u0995\u09be\u09b2 \u09b8\u0995\u09be\u09b2 \u09e7\u09e6\u099f\u09be\u09df \u0986\u09aa\u09a8\u09be\u09b0 \u0985\u09b0\u09cd\u09a1\u09be\u09b0 \u09aa\u09cc\u0981\u099b\u09c7 \u09af\u09be\u09ac\u09c7\u0964"  #@param {type:"string"}
LANG  = "bn"  #@param ["as","bn","en","gu","hi","kn","ml","mr","or","pa","ta","te"]
VOICE = "Jahnavi"  #@param ["Ahana","Vaidehi","Malini","Charvi","Jahnavi","Anirban","Nakul","Vasant","Siddhant","Satvik","Abhay","Nivedita","Ishwari","Ritvik","Vrinda","Sreekar","Kayal","Shaunak","Akshat","Tanushree"]
OUT   = "speech.wav"  #@param {type:"string"}

audio = tts.generate(TEXT, lang=LANG, voice=VOICE, normalize=True)  # float32 mono, 24 kHz
sf.write(OUT, audio, 24000)
print(f'{len(audio) / 24000:.2f} s  ->  {OUT}')
ipd.display(ipd.Audio(OUT))

## Level it

The release's own 12 samples sit between **-21.6 and -27.8 LUFS** — 6.2 LU of spread between voices. If this
speech is going into a mix at a fixed dialogue level, normalise per utterance rather than trusting the model
to be consistent. Targets: **-23 LUFS** (EBU R128), **-24 LKFS** (Indian TV / ATSC A-85), **-14 LUFS**
(YouTube, Spotify).

Output is **24 kHz mono** — 12 kHz of audio bandwidth, and the samples do run flat to Nyquist. That is fine
for dialogue, but it is not a 48 kHz deliverable: upsampling to a 48 k timeline does not put the top octave
back, it only stops the sample rates fighting.

In [ ]:
#@title 5 · Normalise to a loudness target
TARGET_LUFS   = -23.0  #@param {type:"number"}
CEILING_DBTP  = -1.0   #@param {type:"number"}

# loudness.py lives next to this notebook in the repo; falls back to a fetch on a bare Colab.
import os, urllib.request
if not os.path.exists('loudness.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/ayanisone/11/claude/ecstatic-rubin-vivnnp/maya2-native/loudness.py',
        'loudness.py')
import loudness

x, sr = sf.read(OUT, dtype='float64')
y, m = loudness.normalize(x, sr, target_lufs=TARGET_LUFS, ceiling_dbtp=CEILING_DBTP)
sf.write('speech_normalized.wav', y, sr)
print(f"{m['lufs_in']:.2f} -> {m['lufs_out']:.2f} LUFS   gain {m['gain_db']:+.2f} dB   "
      f"true peak {m['dbtp_out']:.2f} dBTP   ceiling-limited: {m['ceiling_limited']}")
ipd.display(ipd.Audio('speech_normalized.wav'))

## Clone a voice

Zero-shot, from 3-10 s of clean single-speaker reference plus its **exact** transcript. Reference language and
output language can differ — a Bengali reference can speak Tamil. Get the speaker's permission first.

Similarity is approximate, and the release reports no dedicated cloning benchmark — noise, multiple speakers,
short references and inaccurate transcripts all degrade it.

In [ ]:
#@title 6 · Register a reference voice and speak as it
REF_AUDIO = "reference.wav"  #@param {type:"string"}
REF_TEXT  = ""  #@param {type:"string"}
REF_LANG  = "bn"  #@param ["as","bn","en","gu","hi","kn","ml","mr","or","pa","ta","te"]
OUT_TEXT  = "\u0986\u09aa\u09a8\u09be\u09b0 \u0985\u09b0\u09cd\u09a1\u09be\u09b0 \u09aa\u09be\u09a0\u09be\u09a8\u09cb \u09b9\u09df\u09c7\u0997\u09c7\u0964"  #@param {type:"string"}
OUT_LANG  = "bn"  #@param ["as","bn","en","gu","hi","kn","ml","mr","or","pa","ta","te"]

assert REF_TEXT.strip(), 'REF_TEXT must be the exact transcript of REF_AUDIO'
tts.register_voice('my_voice', ref_audio=REF_AUDIO, ref_text=REF_TEXT, lang=REF_LANG)
cloned = tts.generate(OUT_TEXT, lang=OUT_LANG, voice='my_voice', normalize=True)
sf.write('cloned.wav', cloned, 24000)
ipd.display(ipd.Audio('cloned.wav'))

In [ ]:
#@title 7 · Stream (16-bit PCM chunks, 80 ms frames)
STREAM_TEXT = "Your order has shipped and will arrive tomorrow."  #@param {type:"string"}

chunks = []
for chunk in tts.stream(STREAM_TEXT, lang='en', voice='Siddhant', normalize=True):
    chunks.append(chunk)            # int16-ready arrays, 24 kHz mono, one per 80 ms frame
pcm = np.concatenate(chunks).astype('<i2')
sf.write('streamed.wav', pcm, 24000, subtype='PCM_16')
print(f'{len(chunks)} frames, {len(pcm) / 24000:.2f} s')
ipd.display(ipd.Audio('streamed.wav'))

In [ ]:
#@title 8 · Release the GPU
tts.close()
print('closed')

## Gotchas

- **`normalize=True` on every Python-API call.** Default is off there, on in the servers.
- **URLs and code-mixed text are the weak spots.** Release CER on web addresses is 29% (20% under their
  revised scorer) against ~3.5% overall. Listen before trusting it with URLs, long digit strings or names.
- **Assamese has no native voice.** No built-in voice has an hour of `as` audio; outputs can drift toward a
  Bengali-like timbre. It is also the weakest language in their benchmark (5.4% CER).
- **Don't put the voice name in the text.** Passing `"Ahana: hello"` is wrong — the engine builds the voice
  prompt itself from `voice=`.
- **Very short turns are unstable** in voice identity. A one- or two-word reply is better padded into a phrase.
- **Emotion tags** (`[laughs]`, `[sighs]`, `[whispers]`) are best-effort and were not a focus of this release.
- **No audio watermark** in this release. Disclose synthetic speech to listeners, and clone only voices you
  have permission to use.
- `max_seats` allocates VRAM up front, busy or not: 7.7 GB at 1 seat, 9.7 GB at 8, 15.8 GB at 32.